# Manually create a Kickbase lineup

Build, validate, review, and select one manual lineup without running an optimizer.


## 1. Imports and shared project logic

The helper module below extracts the optimizer's non-solver data, mapping, and validation conventions.


In [6]:
from __future__ import annotations

import sys
from pathlib import Path

from IPython.display import display


def locate_project_root() -> Path:
    starts = []
    notebook_path = globals().get('__vsc_ipynb_file__')
    if isinstance(notebook_path, str) and notebook_path.strip():
        starts.append(Path(notebook_path).expanduser().resolve().parent)
    starts.append(Path.cwd().resolve())
    for start in starts:
        for candidate in (start, *start.parents):
            if (candidate / 'project_paths.py').is_file():
                return candidate
    raise FileNotFoundError('Could not locate project_paths.py. Start Jupyter from the project root.')


PROJECT_ROOT = locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from manual_lineup_helpers import (
    EXPECTED_POINTS_DIR,
    canonical_arena,
    discover_latest_score,
    format_score,
    lineup_summary_table,
    load_matchday_matches,
    prepare_optimization_data,
    request_arena,
    request_captain,
    request_formation,
    request_matchday,
    request_players_by_position,
    snapshot_players,
    sorted_lineup_indices,
    validate_manual_lineup,
)
from selected_lineups import make_selected_lineup, select_lineup_interactively


## 2. Player pool and expected-points input


In [7]:
metadata = discover_latest_score(EXPECTED_POINTS_DIR)
print('Selected expected-points input:')
print(f'  File: {metadata.path}')
print(f'  Method: {metadata.method}')
print(f'  Metric creation: {metadata.metric_creation_timestamp}')

matchday = request_matchday()
matches, match_source, match_path = load_matchday_matches(matchday)
prepared, mapped_matches, mapping_table = prepare_optimization_data(metadata.path, matches)
print(f'\nValidated player pool: {len(prepared.df):,} players')
print(f'Market-value unit: {prepared.value_unit}; validation uses euros.')
display(mapping_table)


Selected expected-points input:
  File: C:\kickbase project\outputs\expected_points\expected_points_20260823_004913_+0200_sofascore_overall_rating_odds_lineup_20260823_005325_+0200.csv
  Method: sofascore_overall_rating_odds_lineup
  Metric creation: 20260823_005325_+0200


Enter the matchday to optimise the squad for:  1


Match source used: SofaScore (C:\kickbase project\outputs\sofascore\match_ids\match_ids_1.json)
Club mapping mode: embedded Kickbase team-ID bridge

Validated player pool: 469 players
Market-value unit: euros; validation uses euros.


,Match ID,JSON Home Team,CSV Home Club,JSON Away Team,CSV Away Club
0,16434087,FC Bayern München,FC Bayern München (2),VfB Stuttgart,VfB Stuttgart (9)
1,16434022,1. FC Köln,1. FC Köln (28),TSG Hoffenheim,TSG Hoffenheim (14)
2,16434026,1. FC Union Berlin,1. FC Union Berlin (40),Eintracht Frankfurt,Eintracht Frankfurt (4)
3,16434044,1. FSV Mainz 05,1. FSV Mainz 05 (18),SC Paderborn 07,SC Paderborn 07 (29)
4,16434023,RB Leipzig,RB Leipzig (43),Borussia M'gladbach,Borussia Mönchengladbach (15)
5,16434020,SV 07 Elversberg,SV 07 Elversberg (77),Bayer 04 Leverkusen,Bayer 04 Leverkusen (7)
6,16434025,Borussia Dortmund,Borussia Dortmund (3),Hamburger SV,Hamburger SV (6)
7,16434029,SC Freiburg,SC Freiburg (5),SV Werder Bremen,SV Werder Bremen (10)
8,16434039,FC Augsburg,FC Augsburg (13),FC Schalke 04,FC Schalke 04 (8)


## 3. Arena and formation


In [8]:
arena = request_arena()
formation, formation_counts = request_formation()
print(f'Using {arena.name}: €{arena.budget_eur:,} budget, max {arena.max_players_per_club} players per club.')
print(f'Formation: {formation}')


Choose arena:
  1. Bundesliga Arena — budget €250,000,000, max 3 per club, max 4 per match
  2. KickbaseKIS Arena — budget €150,000,000, max 2 per club, max 4 per match


Arena number or name:  1
Formation (4-4-2, 4-2-4, 3-4-3, 4-3-3, 5-3-2, 3-5-2, 5-4-1, 4-5-1, 3-6-1, 5-2-3):  5-4-1


Using Bundesliga Arena: €250,000,000 budget, max 3 players per club.
Formation: 5-4-1


## 4. Choose players by position and select a captain


In [9]:
selected_indices = request_players_by_position(prepared, formation_counts)
captain_index = request_captain(prepared, selected_indices)
print(f"Captain: {prepared.df.loc[captain_index, prepared.columns['player_name']]}")


GK 1 of 1:  Nahuel Noll


Added Nahuel Noll.



DEF 1 of 5:  Julian Ryerson


Added Julian Ryerson.



DEF 2 of 5:  Ramy Bensebaini


...

Added Ezechiel Banzuzi.



FOR 1 of 1:  Johan Bakayoko


Added Johan Bakayoko.



Captain name:  Michael Olise


Captain: Michael Olise


## 5. Validate arena rules and handle a budget-only override


In [10]:
validation = validate_manual_lineup(
    prepared, selected_indices, formation, arena, mapped_matches, captain_index
)
budget_override_used = False
if not validation['budget_valid']:
    total_value = int(validation['total_value_eur'])
    excess = int(validation['budget_excess_eur'])
    while True:
        answer = input(
            f'The squad costs €{total_value:,}; the {arena.name} budget is €{arena.budget_eur:,}; '
            f'it exceeds the budget by €{excess:,}. Does this squad work in your actual game? [y/n]: '
        ).strip().casefold()
        if answer in {'y', 'yes'}:
            budget_override_used = True
            break
        if answer in {'n', 'no'}:
            break
        print('Please answer yes or no.')

lineup_valid = bool(validation['non_budget_valid']) and (
    bool(validation['budget_valid']) or budget_override_used
)


The squad costs €257,189,203; the Bundesliga Arena budget is €250,000,000; it exceeds the budget by €7,189,203. Does this squad work in your actual game? [y/n]:  y


## 6. Final validation summary


In [11]:
sorted_indices = sorted_lineup_indices(prepared, selected_indices)
summary_table = lineup_summary_table(prepared, sorted_indices, captain_index)
print('=== Manual lineup summary ===')
print(f'Arena: {arena.name}')
print(f'Formation: {formation}')
print(f'Matchday: {matchday}')
print(f'Captain: {prepared.df.loc[captain_index, prepared.columns["player_name"]]}')
print(f"Total squad cost: €{int(validation['total_value_eur']):,}")
print(f"Total expected points (captain doubled): {format_score(int(validation['total_score_units']), prepared.score_scale)}")
print(f"Budget override used: {'Yes' if budget_override_used else 'No'}")
print('\nValidation checks:')
for check in validation['checks']:
    passed = bool(check['passed']) or (bool(check['budget']) and budget_override_used)
    status = 'PASS' if passed else 'FAIL'
    suffix = ' (manual override)' if bool(check['budget']) and budget_override_used else ''
    print(f"  [{status}] {check['rule']}: {check['details']}{suffix}")
print(f"\nOverall result: {'VALID' if lineup_valid else 'INVALID'}")
display(summary_table)


=== Manual lineup summary ===
Arena: Bundesliga Arena
Formation: 5-4-1
Matchday: 1
Captain: Michael Olise
Total squad cost: €257,189,203
Total expected points (captain doubled): 134.499754
Budget override used: Yes

Validation checks:
  [PASS] Formation and squad size: actual={'DEF': 5, 'MID': 4, 'GK': 1, 'FOR': 1}; expected={'GK': 1, 'DEF': 5, 'MID': 4, 'FOR': 1}; players=11/11
  [PASS] Unique players: all player IDs are unique
  [PASS] Captain: Michael Olise
  [PASS] Budget: €257,189,203 / €250,000,000 (exceeds by €7,189,203) (manual override)
  [PASS] Players per club: within limit
  [PASS] Players per match: within limit

Overall result: VALID


,Player,ID,Position,Club,Price,Expected points,Captain
0,Nahuel Noll,2977,GK,SC Paderborn 07,"€12,680,267",0.0,
1,Julian Ryerson,2395,DEF,Borussia Dortmund,"€25,926,775",17.286749,
2,Ramy Bensebaini,2736,DEF,Borussia Dortmund,"€21,726,972",15.993183,
3,Miguel Gutiérrez,3759,DEF,Bayer 04 Leverkusen,"€20,385,787",13.776144,
4,Vladimír Coufal,7745,DEF,TSG Hoffenheim,"€25,768,123",10.378417,
5,Anthony Caci,3470,DEF,1. FSV Mainz 05,"€10,416,096",0.0,
6,Michael Olise,8329,MID,FC Bayern München,"€64,765,581",18.580415,Yes
7,Konstantinos Karetsas,16378,MID,Borussia Dortmund,"€23,538,072",17.098594,
8,Ezechiel Banzuzi,11008,MID,RB Leipzig,"€7,697,909",10.999312,
9,Nadiem Amiri,1639,MID,1. FSV Mainz 05,"€33,544,900",0.0,


## 7. Select and save the valid lineup

Saving replaces only this arena's canonical selected-lineup JSON after confirmation.


In [12]:
if not lineup_valid:
    print('Lineup was not saved because one or more non-overridable arena rules failed.')
    selected_lineup_path = None
else:
    snapshot = make_selected_lineup(
        league=arena.name,
        players=snapshot_players(prepared, sorted_indices, captain_index),
        expected_points={
            'value': format_score(int(validation['total_score_units']), prepared.score_scale),
            'label': 'Total expected points (captain doubled)',
            'includes_captain_bonus': True,
        },
        source='manual',
        metadata={
            'score_input_file': str(metadata.path),
            'score_method': metadata.method,
            'metric_creation_timestamp': metadata.metric_creation_timestamp,
            'matchday': matchday,
            'formation': formation,
            'budget_override_used': budget_override_used,
            'nominal_budget_eur': arena.budget_eur,
            'total_value_eur': int(validation['total_value_eur']),
        },
    )
    selected_lineup_path = select_lineup_interactively(snapshot)


Select this lineup for Bundesliga Arena? [y/n]:  y


Current selected lineup for Bundesliga Arena: expected points=187.746417, players=[Manuel Neuer, Julian Ryerson, Willi Orban, Ramy Bensebaini, Nathaniel Brown, Konstantinos Karetsas, Aleksandar Pavlović, Brajan Gruda, Nicolas Seiwald, Yannik Engelhardt, Afonso Moreira]


Replace the current selected lineup? [y/n]:  y


Selected lineup saved to: C:\kickbase project\outputs\selected_lineups\bundesliga-arena.json
